# 一、PyTorch Tensor 核心详解
#### Tensor 开门见山核心定位
**Tensor（张量）是 PyTorch 的核心对象，本质是支持 GPU、自动微分的多维数值数组**，整个深度学习框架所有运算、数据存储全部依赖它。

#### 1.1 全局核心地位
1. 所有输入样本（图像/文本/时序）必须转为 Tensor 才能参与 PyTorch 的运算；
2. 模型权重、偏置、梯度底层全部存储为 Tensor；
3. 前向推理、反向求导、参数更新、GPU加速、分布式训练全部基于 Tensor；
4. 没有 Tensor，PyTorch 无法执行任何深度学习计算。

#### 1.2 Tensor 四大核心能力
1. 多维数组存储：对标 NumPy ndarray，支持 0~N 维数值容器；
2. Autograd 自动微分：`requires_grad=True` 让张量被追踪，引擎自动记录每一步运算，一次 `backward()` 就反向算出梯度——无需手动推导；
3. 跨硬件加速：一套API无缝切换 CPU / GPU / MPS；
4. 深度学习生态载体：Tensor 本身不直接提供卷积/损失等方法，但它是 `torch.nn` 卷积层、池化层、损失函数、分布式算子的统一数据载体与运算基础。

#### 学习目标
1. 彻底搞懂：Tensor 的本质——多维数值数组对象
2. 核心思考：已有 NumPy ndarray，为什么深度学习单独设计 Tensor？
3. 分段代码实例，逐条掌握 Tensor 独有属性、独有方法，对比 ndarray 缺失能力
4. 理清 Tensor ↔ NumPy ndarray 双向转换、深浅拷贝区别
5. 掌握：张量创建、四则运算、矩阵运算、形状变换、随机数、设备迁移、序列化保存加载
6. 分类梳理 Tensor 高频属性与操作，每段独立运行查看输出

#### 前置依赖安装
```bash
pip install torch numpy jupyter
```

In [ ]:
# 导入基础依赖
import torch
import numpy as np
import os
import time
import torch.nn as nn

# 打印版本与设备信息
print(f"PyTorch版本: {torch.__version__}")
print(f"NumPy版本: {np.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"可用计算设备: {device}")

### 二、Tensor 的本质：多维数值数组对象
> [<a href="http://127.0.0.1:5500/html/02_tensor_01_dimension.html" target="_blank">👉 在浏览器打开 张量维度图解</a>](http://127.0.0.1:5500/html/02_tensor_01_dimension.html)

#### 2.1 张量维度定义
- 0维：标量（单个数字）
- 1维：向量（一维列表）
- 2维：矩阵（二维表格）
- 3维：图像 [C, H, W] 通道、高、宽
- 4维：批量图像 [batch, C, H, W] 深度学习标准输入

底层和 NumPy ndarray 同为连续内存多维数组，基础切片、广播语法相似；
但 Tensor 额外搭载自动微分、GPU、网络算子等深度学习专属功能。

#### 2.2 分段演示：不同维度张量创建（基础示例）

In [ ]:
# 0维张量（标量）
t0 = torch.tensor(5)
print("0维张量：", t0)
print("维度数 dim()：", t0.dim())

In [ ]:
# 1维张量（向量）
t1 = torch.tensor([1, 2, 3])
print("1维张量：", t1)
print("维度数 dim()：", t1.dim())

In [ ]:
# 2维张量（矩阵）
t2 = torch.tensor([[1,2],[3,4]])
print("2维张量：")
print(t2)
print("维度数 dim()：", t2.dim())

In [ ]:
# 3维张量（单张图像 [通道,高,宽]）
t3 = torch.rand(2, 28, 28)
print("3维张量 shape：", t3.shape)
print("维度数 dim()：", t3.dim())

In [ ]:
# 4维张量（批量图像 [batch,通道,高,宽]）
t4 = torch.rand(8, 3, 32, 32)
print("4维批量图像张量 shape：", t4.shape)

#### 2.3 张量创建完整分类

> **设计哲学**：PyTorch 提供多种创建方式，每种服务于不同场景。

创建方式分为 **6 大类**：

| 分类 | 方法 | 特点 |
|------|------|------|
| **1. 从数据创建** | `torch.tensor(data)` | 从 Python 列表/NumPy 数组创建，**默认拷贝数据** |
| **2. NumPy 共享** | `torch.from_numpy(ndarray)` | 与 ndarray **共享内存**，修改互相影响 |
| **3. 填充创建** | `zeros` / `ones` / `full` / `empty` | 指定形状，填充固定值或未初始化 |
| **4. 序列创建** | `arange` / `linspace` / `logspace` | 生成等差数列/等比数列 |
| **5. 克隆拷贝** | `tensor.clone()` | **深拷贝**，完全复制数据 + **保留梯度历史** |
| **6. 像类创建** | `zeros_like` / `ones_like` / `full_like` | 参考已有张量的 shape/dtype/device 创建新张量 |

> 说明：第 5 类的 `clone()` 只是"创建方式"之一，这里先记住它的定位——**深拷贝数据，且保留梯度历史**。至于它为什么能保留梯度、与 `detach()` / `torch.tensor()` 有何本质区别、以及有哪些实用场景，放到 **附录二：`clone()` 完整专题** 系统展开，此处不打断"创建方式"这条主线。

In [ ]:
# ============================================================
# 【分类1】torch.tensor() —— 从数据创建（深拷贝）
# ============================================================

t_from_list = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
print("torch.tensor 从列表创建:", t_from_list, t_from_list.dtype)

arr = np.array([4.0, 5.0, 6.0])
t_from_np_copy = torch.tensor(arr)
arr[0] = 999
print("torch.tensor 从 NumPy 创建（深拷贝，不受影响）:", t_from_np_copy)

In [ ]:
# ============================================================
# 【分类2】torch.from_numpy() —— NumPy 零拷贝共享
# ============================================================

arr_shared = np.array([10, 20, 30])
t_shared = torch.from_numpy(arr_shared)
print(f"原始 NumPy: {arr_shared}")
print(f"共享 Tensor: {t_shared}")

arr_shared[1] = 999
print(f"\n修改 NumPy 后 Tensor 同步变化: {t_shared}")

t_shared[2] = 888
print(f"修改 Tensor 后 NumPy 同步变化: {arr_shared}")

In [ ]:
# ============================================================
# 【分类3】填充类 —— zeros / ones / full / empty
# ============================================================

t_zeros = torch.zeros(2, 3)
t_ones = torch.ones(2, 3)
t_full = torch.full((2, 3), fill_value=5.0)
t_empty = torch.empty(2, 3)

print("zeros:\n", t_zeros)
print("ones:\n", t_ones)
print("full(5.0):\n", t_full)
print("empty (未初始化):\n", t_empty)

In [ ]:
# ============================================================
# 【分类4】序列类 —— arange / linspace / logspace
# ============================================================

t_arange = torch.arange(0, 10, step=2)
t_linspace = torch.linspace(0, 1, 5)
t_logspace = torch.logspace(0, 2, 4)

print(f"arange(0, 10, step=2): {t_arange}")
print(f"linspace(0, 1, 5): {t_linspace}")
print(f"logspace(0, 2, 4): {t_logspace}")

In [ ]:
# ============================================================
# 【分类5】clone() —— 深拷贝（保留梯度历史）
# ============================================================

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2
z = y.sum()
z.backward()

x_clone = x.clone()
y_clone = y.clone()

print(f"原始 x: {x}, requires_grad={x.requires_grad}")
print(f"x.clone(): {x_clone}, requires_grad={x_clone.requires_grad}")
print(f"y (中间节点): {y}, grad_fn={y.grad_fn}")
print(f"y.clone(): {y_clone}, grad_fn={y_clone.grad_fn} ← 保留了 grad_fn")

x_clone[0] = 999
print(f"\n修改 clone 后，原 x 不变: {x}")

y_detach = y.detach()
print(f"\ny.detach(): {y_detach}, grad_fn={y_detach.grad_fn} ← 被切断")

In [ ]:
# ============================================================
# 【分类6】*_like 类 —— 按模板创建
# ============================================================

template = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32, device='cpu')

t_zeros_like = torch.zeros_like(template)
t_ones_like = torch.ones_like(template)
t_full_like = torch.full_like(template, fill_value=7)

print(f"模板张量:\n{template}")
print(f"zeros_like:\n{t_zeros_like}")
print(f"ones_like:\n{t_ones_like}")
print(f"full_like(fill=7):\n{t_full_like}")

print("\n💡 *_like 会继承 template 的 dtype 和 device")

#### 2.4 基础四则运算 + 广播机制
> [<a href="http://127.0.0.1:5500/html/02_tensor_02_broadcast.html" target="_blank">👉 在浏览器打开 广播机制图解</a>](http://127.0.0.1:5500/html/02_tensor_02_broadcast.html)


In [ ]:
# 四则运算
a = torch.tensor([1,2,3])
b = torch.tensor([4,5,6])
print("a + b:", a + b)
print("a - b:", a - b)
print("a * b(逐元素乘):", a * b)
print("a / b:", a / b)
print("pow:", torch.pow(a,2))

In [ ]:
# 广播机制演示
mat = torch.tensor([[1,2],[3,4]])
vec = torch.tensor([10,20])
res = mat + vec
print("矩阵 + 向量广播结果:\n", res)
print("广播规则：vec [10,20] → [[10,20],[10,20]]")

#### 2.5 矩阵运算，多种乘法辨析
- `*` ：逐元素乘法 Hadamard
- `@` / torch.matmul()：矩阵乘法，支持高维批量
- `torch.mm()`：仅限2维矩阵乘法，不支持批量

In [ ]:
m1 = torch.tensor([[1,2],[3,4]])
m2 = torch.tensor([[5,6],[7,8]])
print("逐元素相乘 *:\n", m1 * m2)
print("矩阵乘法 @:\n", m1 @ m2)
print("matmul等价:\n", torch.matmul(m1, m2))
print("mm等价(仅2维):\n", torch.mm(m1, m2))

#### 2.6 张量形状变换详解
> [<a href="http://127.0.0.1:5500/html/02_tensor_03_memory.html" target="_blank">👉 在浏览器打开 内存连续性图解</a>](http://127.0.0.1:5500/html/02_tensor_03_memory.html)


> **设计哲学**：深度学习中的形状变换本质是**内存布局的重解释**。PyTorch 提供两类形状变换：
- **视图类（View）**：共享内存，零拷贝，要求内存连续
- **拷贝类（Copy）**：独立内存，可处理不连续内存，有性能开销

> **符号说明**：✅ = 是/具备，❌ = 否/不具备

| 方法 | 是否共享内存 | 要求连续 | 适用场景 |
|------|------------|---------|---------|
| `view()` | ✅ | ✅ | 内存连续时的高效重塑 |
| `reshape()` | ✅* | ❌ | 通用重塑，自动处理不连续 |
| `transpose()` | ✅ | ❌ | 交换维度 |
| `permute()` | ✅ | ❌ | 任意维度重排 |
| `flatten()` | ✅* | ❌ | 展平指定维度 |
| `contiguous()` | ❌* | ✅ | 将不连续张量转为连续 |

> \* `reshape()` 和 `flatten()` 尽可能共享内存，不连续时可能触发拷贝；`contiguous()` 可能拷贝。

In [ ]:
# 2.6.1 view() vs reshape() —— 核心区别

t = torch.arange(12).reshape(3, 4)
print("原始张量:\n", t)

t_view = t.view(4, 3)
print(f"\nview(4,3) shape: {t_view.shape}")

t_transposed = t.transpose(0, 1)
print(f"\ntranspose 后是否连续: {t_transposed.is_contiguous()}")
t_reshape = t_transposed.reshape(3, 4)
print(f"reshape 不连续张量成功: {t_reshape.shape}")

try:
    t_view_err = t_transposed.view(3, 4)
except RuntimeError as e:
    print(f"\n❌ view 处理不连续报错: {e}")

t_contiguous = t_transposed.contiguous()
t_view_ok = t_contiguous.view(3, 4)
print(f"\n✅ contiguous() 后 view 成功: {t_view_ok.shape}")

In [ ]:
# 2.6.2 transpose() vs permute() —— 维度交换

t3d = torch.randn(2, 3, 4)
print(f"原始 3D shape: {t3d.shape}")
print(f"transpose(0,1) → {t3d.transpose(0,1).shape}")
print(f"transpose(1,2) → {t3d.transpose(1,2).shape}")
print(f"permute(2,0,1) → {t3d.permute(2,0,1).shape}")

# Transformer 维度变换
batch, seq, hidden = 2, 10, 512
num_heads, head_dim = 8, 64
x = torch.randn(batch, seq, hidden)
x_reshaped = x.view(batch, seq, num_heads, head_dim)
x_transposed = x_reshaped.transpose(1, 2)
print(f"\nTransformer: {x.shape} → {x_reshaped.shape} → {x_transposed.shape}")

In [ ]:
# 2.6.3 unsqueeze() vs squeeze() —— 增删维度

t = torch.tensor([1, 2, 3])
print(f"原始 1D: {t.shape}")
print(f"unsqueeze(0): {t.unsqueeze(0).shape}")
print(f"unsqueeze(1): {t.unsqueeze(1).shape}")

t_squeeze = t.unsqueeze(0).unsqueeze(2)
print(f"\nunsqueeze 后: {t_squeeze.shape}")
print(f"squeeze() 全部移除: {t_squeeze.squeeze().shape}")

single_image = torch.randn(3, 224, 224)
batch_image = single_image.unsqueeze(0)
print(f"\n单张图像 {single_image.shape} → 批处理 {batch_image.shape}")

In [ ]:
# 2.6.4 flatten() —— 展平操作

t = torch.arange(24).reshape(2, 3, 4)
print(f"原始 3D: {t.shape}")
print(f"flatten(): {t.flatten().shape}")
print(f"flatten(1): {t.flatten(1).shape}")
print(f"flatten(1,2): {t.flatten(1,2).shape}")

batch, C, H, W = 4, 64, 8, 8
features = torch.randn(batch, C, H, W)
flattened = features.flatten(1)
print(f"\nCNN 特征图展平: {features.shape} → {flattened.shape}")

#### 2.7 张量高级索引与切片

In [ ]:
# 2.7.1 整数索引与切片

t = torch.arange(24).reshape(2, 3, 4)
print(f"原始张量 shape: {t.shape}")
print("t[0]:\n", t[0])
print("t[:, 0]:\n", t[:, 0])
print("t[0, 1, 2]:", t[0, 1, 2])
print("t[:, :, 1:3]:\n", t[:, :, 1:3])

In [ ]:
# 2.7.2 布尔索引（条件筛选）

t = torch.randn(5)
print(f"随机张量: {t}")

mask = t > 0
print(f"大于 0 的掩码: {mask}")
print(f"筛选结果: {t[mask]}")

t2d = torch.randn(3, 4)
mask2d = (t2d > 0) & (t2d < 1)
print(f"\n2D 张量筛选 0~1 之间的元素:\n{t2d[mask2d]}")

In [ ]:
# 2.7.3 花式索引（Fancy Indexing）

t = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(f"原始:\n{t}")

rows = torch.tensor([0, 2])
print(f"\n取第 0 行和第 2 行:\n{t[rows]}")

cols = torch.tensor([2, 0])
print(f"取 (0,2) 和 (2,0): {t[rows, cols]}")

logits = torch.randn(10, 5)
indices = torch.tensor([0, 3, 1, 4, 2, 0, 3, 1, 4, 2])
selected = logits[torch.arange(10), indices]
print(f"\n按索引采样: {selected.shape}")

### 三、自动微分：Tensor 相比 NumPy 的核心优势
> [<a href="http://127.0.0.1:5500/html/02_tensor_04_graph.html" target="_blank">👉 在浏览器打开 计算图与梯度流图解</a>](http://127.0.0.1:5500/html/02_tensor_04_graph.html)


#### 3.1 NumPy ndarray 的长处与短板

**NumPy ndarray 的优势**：
1. C 底层实现，基础运算快于原生 Python 列表；
2. 多维数组、广播、切片、矩阵运算、统计聚合齐全；
3. 数据分析、传统机器学习生态完善。

**NumPy ndarray 在深度学习中的致命短板**：
无自动微分、无 GPU 调度、无卷积/损失算子、无分布式接口——这些正是深度学习的核心需求。

> **核心疑问**：已有 NumPy ndarray，为什么还要 Tensor？
> 
> **答案**：Tensor = ndarray 的数组能力 + 自动微分 + GPU 加速 + 深度学习算子。

---

#### 模块1：Tensor 承载着 Autograd 自动微分

**什么是 Autograd？**

Autograd（**Auto**matic **grad**ient，自动微分）是 PyTorch 的核心引擎，它自动跟踪张量上的所有运算，并在反向传播时自动计算梯度。

> **一句话理解**：你只管写前向传播（`y = w * x`），PyTorch 的 Autograd 引擎自动帮你算反向传播（`dy/dw` 和 `dy/dx`）。

**工作流程：**

```
前向传播                    反向传播（Autograd 自动完成）
─────────────────────────────────────────────────────────────
x (数据叶子) ─┐
             ├──► y = w * x ──► loss ──► loss.backward()
w (参数叶子) ─┘                              │
                                            ▼
                                 自动计算 dw 和 dx
                                 存入 w.grad 和 x.grad
```

**Autograd 自动完成的三个任务：**
1. **构建计算图**：记录每个运算的 `grad_fn`（如 `MulBackward`）
2. **链式求导**：从 `loss` 开始，沿 `grad_fn` 链反向传播
3. **存储梯度**：将计算结果存入每个叶子节点的 `.grad` 属性

> 整个过程中，你**只需要调用一次 `.backward()`**，其他全部由 Autograd 引擎自动完成。

**为什么能"自动"？** 一句话：**前向留痕，反向消费**。
你设置的 `requires_grad` 是追踪开关，`grad_fn` 等属性是引擎写回的"痕迹"——
每做一次运算，引擎就记下一个节点、存好依赖与求导规则；`backward()` 时再顺着这些痕迹倒着走一遍，
把梯度写进 `.grad`。所以属性是你和引擎打交道的接口，真正干活的是"建图 + 遍历"这套机制。
> 本章先会用即可；这套机制的内部细节，下一章《torch.autograd 核心详解》会彻底讲透。

#### Tensor 身上与自动求导相关的属性和方法

> 下面这些属性和方法都是 **Tensor 对象自带的**，专门服务于自动求导。本章只要求**认识它们、知道怎么用**；至于底层机制（计算图怎么建、梯度怎么沿链路回传、组合规则如何判定），是下一章《torch.autograd 核心详解》的主线——这一节就是为它做的准备。

**① 核心属性**（与自动求导相关）：

| 属性 | 谁设置 | 含义 | 下一章讲透什么 |
|------|--------|------|---------------|
| `requires_grad` | 用户指定（或继承） | 是否需要梯度；开了才会被追踪 | 传播规则 |
| `grad` | autograd 写入 | 反向结束后存放梯度（叶子默认保留） | 谁保留、谁丢弃 |
| `grad_fn` | autograd 写入 | 记录"谁造了我"，反向遍历的入口 | 链路的构建与遍历 |
| `is_leaf` | 系统自动判定 | 是否用户直接创建的叶子张量 | 四种组合与判定逻辑 |
| `_version` | 系统维护 | 版本计数，检测"反向所需数据被原地改动" | 版本检查机制 |

**② 方法**（Tensor 自带，用来操控自动求导）：

| 方法 | 作用 | 下一章讲透什么 |
|------|------|---------------|
| `requires_grad_()` | 原地开关追踪 | 叶子/非叶子的调用限制 |
| `backward()` | 触发反向，梯度写入 `.grad` | 遍历算法、多分支累加 |
| `detach()` | 切断追踪（与原张量共享内存） | 与 `no_grad()` 的本质区别 |
| `retain_grad()` | 保留中间张量的梯度 | 梯度去向管理 |
| `register_hook(fn)` | 注册梯度钩子 | 梯度调试技巧 |
| `grad.zero_()` | 清空梯度槽位 | 梯度累积与清零策略 |

**Tensor 的自动微分机制**：

1. **创建叶子张量**时，设置 `requires_grad=True`，表示需要对该张量求梯度——**从此处开始构建计算图**。
2. 调用 `loss.backward()` 时，Autograd 引擎从 `grad_fn` 开始，沿计算图反向遍历，链式求导并计算梯度。
3. 梯度计算完成后，**叶子张量**的梯度会存入其 `.grad` 属性；非叶子张量的梯度默认被释放（可通过 `retain_grad()` 保留）。

---

#### 深入理解：`requires_grad` 与 `is_leaf` 的关系

> 本部分系统梳理两个关键属性的所有合法/非法组合，及背后的计算图设计逻辑。

**核心定义：**
- `is_leaf`：判断张量是否为**叶子节点**（由用户直接创建，而非运算产生），**与 `requires_grad` 的值无关**。该值由 PyTorch 系统**自动判定**，用户无法直接指定。
- `requires_grad`：判断张量是否**需要对该张量求梯度**（参与反向传播计算）。该值由用户通过构造参数或 `requires_grad_()` 指定。

**叶子节点的两种类型：**

用一个最简单的线性表达式来理解：**`y = w * x`**
- **数据叶子**：`x`（输入样本，不求导）
- **参数叶子**：`w`（模型权重，求导）
- **中间节点**：`y`（运算结果，自动继承梯度）

| 类型 | `requires_grad` | 典型场景 | 对应 `y = w * x` 中的角色 | 示例 |
|:---|:---:|:---|:---|:---|
| **数据叶子** | `False` | 输入样本（图像/文本/时序） | **`x`** | `x = torch.randn(3, 224, 224)` |
| **参数叶子** | `True` | 模型权重、偏置 | **`w`** | `w = torch.randn(224, 10, requires_grad=True)` |
| **中间节点** | `True`（系统自动继承） | 前向传播结果 | **`y`** | `y = x @ w` |

> **核心认知**：`is_leaf=True` 只表示"用户直接创建"，**不表示"需要梯度"**。数据叶子（不求导）和参数叶子（求导）都是合法的叶子节点。

**所有组合状态一览表：**

| `requires_grad` | `is_leaf` | 是否合法存在 | 叶子类型 | 对应 `y = w * x` 中的角色 | 如何产生 |
| :---: | :---: | :---: | :---: | :--- | :--- |
| `True` | `True` | ✅ 合法 | **参数叶子** | **`w`**（权重） | 用户直接创建并指定 `requires_grad=True` |
| `False` | `True` | ✅ 合法 | **数据叶子** | **`x`**（输入） | 用户直接创建，未指定 `requires_grad`（默认为 `False`） |
| `True` | `False` | ✅ 合法（**系统自动产生**） | 中间节点 | **`y`**（输出） | `x @ w` 等运算产生，自动继承梯度 |
| **`False`** | **`False`** | ❌ **非法，永不出现** | — | — | — |

**关键认知：**

1. **`is_leaf=True` 的张量，`requires_grad` 可以是 `True` 或 `False`。** 叶子节点只代表"用户直接创建"，不代表"一定要求导"。

2. **`is_leaf=False` 的张量，`requires_grad` 必须为 `True`。** 如果一个张量是运算产生的（非叶子），它要么：
   - 参与后续运算，继续传递梯度 → `requires_grad=True`
   - 不参与后续运算，是最终结果 → 系统会将其优化为常量，`is_leaf` 被重新标记为 `True`

3. **`requires_grad=False, is_leaf=False` 为什么不存在？**
   - `is_leaf=False` 意味着该张量是**运算的输出**。
   - 如果所有输入都不需要梯度，PyTorch 不会为该运算构建计算图，输出直接被当作叶子节点（`is_leaf=True`）。
   - 如果有任一输入需要梯度，输出会自动 `requires_grad=True`。
   - 因此 `False + False` 在任何情况下都无法形成。

**代码验证：数据叶子 vs 参数叶子 —— `y = w * x` 实例**

In [ ]:
# 数据叶子 vs 参数叶子 —— 用 y = w * x 理解
print("=" * 60)
print("数据叶子 vs 参数叶子 —— y = w * x")
print("=" * 60)

print("\n1. 数据叶子 x（输入，不求导）:")
# 模拟 MNIST 单张图像输入（灰度图 1x28x28 → 展平为 784 维）
x = torch.randn(3, 784)
print(f"   x: shape={x.shape}, requires_grad={x.requires_grad}, is_leaf={x.is_leaf}")

print("\n2. 参数叶子 w（权重，求导）:")
# 模拟 线性层权重 [输入维度 784 → 输出维度 256]
w = torch.randn(784, 256, requires_grad=True)
print(f"   w: shape={w.shape}, requires_grad={w.requires_grad}, is_leaf={w.is_leaf}")

print("\n3. 中间节点 y（运算产生，自动继承梯度）:")
y = x @ w  # (3, 256)
print(f"   y: shape={y.shape}, requires_grad={y.requires_grad}, is_leaf={y.is_leaf}")
print(f"   y.grad_fn = {y.grad_fn} (记录了 'Mm' 矩阵乘法运算)")

print("\n" + "-" * 60)
print("💡 总结：")
print("   • 数据叶子 x: 用户直接创建（torch.randn），不求导（requires_grad=False）")
print("   • 参数叶子 w: 用户直接创建（torch.randn），求导（requires_grad=True）")
print("   • 中间节点 y: 运算产生（x @ w），自动继承梯度（requires_grad=True）")
print("   • is_leaf 与 requires_grad 正交，只有 False+False 不存在")

In [ ]:
# 验证四种状态
print("\n" + "=" * 60)
print("requires_grad 与 is_leaf 关系验证")
print("=" * 60)

print("\n1. True + True（参数叶子 → w）:")
w = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)
print(f"   requires_grad={w.requires_grad}, is_leaf={w.is_leaf} ✅")

print("\n2. False + True（数据叶子 → x）:")
x = torch.tensor([1.0, 2.0, 3.0])
print(f"   requires_grad={x.requires_grad}, is_leaf={x.is_leaf} ✅")

print("\n3. True + False（运算产生，继承梯度 → y = x * w）:")
y = x * w
print(f"   requires_grad={y.requires_grad}, is_leaf={y.is_leaf}, grad_fn={y.grad_fn} ✅")

print("\n4. False + False（理论上不存在）:")
a = torch.tensor([1.0, 2.0, 3.0])  # False + True
b = torch.tensor([4.0, 5.0, 6.0])  # False + True
c = a + b  # 所有输入都不需要梯度
print(f"   a + b 的输出: requires_grad={c.requires_grad}, is_leaf={c.is_leaf}")
print(f"   → 系统自动优化为叶子节点，is_leaf=True ❌ 不会出现 False+False")

In [ ]:
# 验证：requires_grad_() 对叶子/非叶子节点的调用规则（非叶子只能由 True 改 False，不能改回 True）
print("\n" + "=" * 60)
print("requires_grad_() 调用合法性验证")
print("=" * 60)

print("\n1. 数据叶子 → 参数叶子（调用 requires_grad_(True)）:")
x = torch.tensor([1.0, 2.0, 3.0])
print(f"   调用前: requires_grad={x.requires_grad}, is_leaf={x.is_leaf}")
x.requires_grad_(True)
print(f"   调用后: requires_grad={x.requires_grad}, is_leaf={x.is_leaf} ✅")
print(f"   → 数据叶子变为参数叶子")

print("\n2. 参数叶子 → 数据叶子（调用 requires_grad_(False)）:")
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"   调用前: requires_grad={w.requires_grad}, is_leaf={w.is_leaf}")
w.requires_grad_(False)
print(f"   调用后: requires_grad={w.requires_grad}, is_leaf={w.is_leaf} ✅")
print(f"   → 参数叶子变为数据叶子")

print("\n3. 中间节点调用 requires_grad_() ❌:")
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2  # 中间节点，requires_grad=True, is_leaf=False
print(f"   y: requires_grad={y.requires_grad}, is_leaf={y.is_leaf}")
try:
    y.requires_grad_(False)  # 可以改成 False
    print(f"   ✅ 改成 False 成功: {y.requires_grad}")
except RuntimeError as e:
    print(f"   ❌ 改成 False 报错: {e}")

try:
    y.requires_grad_(True)
except RuntimeError as e:
    print(f"   ❌ 改成 True 报错: {e}")

---

#### Tensor 的 Autograd 代码演示

In [ ]:
# 1. 开启梯度追踪
x = torch.tensor(3.0, requires_grad=True)
print(f".requires_grad: {x.requires_grad}")
print(f".is_leaf: {x.is_leaf}")

y = x ** 2 + 2 * x
mid_node = x * 2
print(f"中间张量 mid_node.is_leaf: {mid_node.is_leaf}")

In [ ]:
# 2. grad_fn 属性
print(f"x.grad_fn = {x.grad_fn} (叶子张量)")
print(f"mid_node.grad_fn = {mid_node.grad_fn}")

In [ ]:
# 3. backward() 自动求导
y.backward()
print(f"dy/dx 梯度 = {x.grad.item()}")

In [ ]:
# 4. .data 属性
x_data = x.data
print(f"原始张量 requires_grad: {x.requires_grad}")
print(f".data 视图 requires_grad: {x_data.requires_grad}")
x_data += 10
print(f"修改.data同步影响原张量数值: {x.item()}")

In [ ]:
# 5. .data 老式SGD更新
w = torch.tensor([1.0], requires_grad=True)
loss = w ** 2
loss.backward()
w.data -= 0.1 * w.grad
print(f"更新后 w = {w.item()}")

In [ ]:
# 6. detach() 官方推荐
x_no_grad = x.detach()
print(f"detach后 requires_grad: {x_no_grad.requires_grad}")

# 关键纠正：detach 视图与原张量【共享内存】，in-place 修改会影响原值
a = torch.tensor(10.0, requires_grad=True)
a_det = a.detach()
a_det += 5
print(f"detach 视图 in-place 修改后，原张量也变了: {a.item()}（应为 15.0）")
print("   → detach 与 .data 一样共享数据；区别在于 detach 保留版本计数、更安全")
print("   → 若想真正隔离数据，需再 .clone()：x.detach().clone()")

In [ ]:
# 7. ndarray 无梯度属性
arr = np.array(3.0)
print("验证 ndarray 缺失梯度能力:")
try:
    arr.requires_grad = True
except AttributeError as e:
    print(f"  ❌ arr.requires_grad: {e}")
try:
    arr.backward()
except AttributeError as e:
    print(f"  ❌ arr.backward(): {e}")
try:
    print(arr.grad)
except AttributeError as e:
    print(f"  ❌ arr.grad: {e}")
try:
    print(arr.grad_fn)
except AttributeError as e:
    print(f"  ❌ arr.grad_fn: {e}")
print("\n✅ 结论：ndarray 完全没有梯度相关的任何属性或方法")

#### 模块2：Tensor 独有 GPU 设备调度能力

In [ ]:
t = torch.tensor([1, 2, 3])
print(f".device: {t.device}")

t_dev = t.to(device)
print(f".to(device): {t_dev.device}")

if torch.cuda.is_available():
    t_gpu = t.cuda()
    print(f".cuda(): {t_gpu.device}")
    t_back = t_gpu.cpu()
    print(f".cpu(): {t_back.device}")

In [ ]:
# ndarray 无设备调度
arr = np.array([1,2,3])
print("验证 ndarray 缺失设备调度:")
try:
    print(arr.device)
except AttributeError as e:
    print(f"  ❌ arr.device: {e}")
try:
    arr.cuda()
except AttributeError as e:
    print(f"  ❌ arr.cuda(): {e}")
try:
    arr.to(device)
except AttributeError as e:
    print(f"  ❌ arr.to(): {e}")
print("\n✅ 结论：ndarray 完全没有设备调度相关的任何属性或方法")

#### 模块3：Tensor 配套深度学习网络算子

In [ ]:
# Tensor 卷积运算
img_t = torch.randn(2, 3, 32, 32)
conv = nn.Conv2d(3, 16, 3, padding=1)
feat = conv(img_t)
print(f"卷积输入: {img_t.shape} → 输出: {feat.shape}")

# ndarray 无法送入卷积层
img_arr = np.random.randn(2, 3, 32, 32)
print("\n验证 ndarray 无法送入深度学习算子:")
try:
    conv_arr = nn.Conv2d(3,16,3)(img_arr)
except TypeError as e:
    print(f"  ❌ ndarray传入Conv2d报错: {e}")
print("\n✅ 结论：ndarray 无法直接用于 PyTorch 深度学习算子")

#### 模块4：分布式训练配套接口
Tensor 原生支持 torch.distributed；NumPy 无原生分布式同步工具。

#### 3.3 GPU加速直观测速对比

In [ ]:
# CPU Tensor 计时
t_cpu = torch.rand(1000, 1000)
start = time.time()
res = t_cpu @ t_cpu.T
print(f"CPU张量矩阵乘法: {(time.time() - start):.4f}s")

# NumPy 计时
arr_cpu = np.random.rand(1000, 1000)
start = time.time()
res = arr_cpu @ arr_cpu.T
print(f"NumPy矩阵乘法: {(time.time() - start):.4f}s")

# GPU Tensor 计时
if torch.cuda.is_available():
    t_gpu = t_cpu.to(device)
    torch.cuda.synchronize()
    start = time.time()
    res = t_gpu @ t_gpu.T
    torch.cuda.synchronize()
    print(f"GPU张量矩阵乘法: {(time.time() - start):.4f}s")

### 四、Tensor 与 NumPy 异同与转换
> [<a href="http://127.0.0.1:5500/html/02_tensor_05_numpy.html" target="_blank">👉 在浏览器打开 NumPy与Tensor内存图解</a>](http://127.0.0.1:5500/html/02_tensor_05_numpy.html)

#### 4.1 异同对比表格

> **符号说明**：✅ = 是/具备，❌ = 否/不具备

| 能力分类 | torch.Tensor | numpy.ndarray |
|------|--------------|---------------|
| 梯度相关属性/方法 | ✅ | ❌ |
| 设备属性/迁移 | ✅ | ❌ |
| 深度学习算子 | ✅ | ❌ |
| 分布式接口 | ✅ | ❌ |

#### 4.2 转换两种模式

In [ ]:
# 场景1：ndarray -> Tensor 共享内存
arr = np.array([10,20,30])
t_shared = torch.from_numpy(arr)
arr[0] = 999
print("共享内存联动: ", t_shared)

In [ ]:
# 场景2：ndarray -> Tensor 深拷贝
arr2 = np.array([1,2,3])
t_copy = torch.tensor(arr2)
arr2[0] = 666
print("深拷贝隔离: ", t_copy)

In [ ]:
# 场景3：Tensor -> ndarray 共享内存
t = torch.tensor([5,6,7])
arr_shared = t.numpy()
t[1] = 888
print("共享数组联动: ", arr_shared)

In [ ]:
# 场景4：Tensor -> ndarray 深拷贝
t2 = torch.tensor([9,8,7])
arr_copy = t2.numpy().copy()
t2[0] = 111
print("拷贝数组隔离: ", arr_copy)

In [ ]:
# 场景5：GPU Tensor 转 ndarray
if torch.cuda.is_available():
    t_gpu = torch.tensor([1,2]).cuda()
    arr_gpu = t_gpu.cpu().numpy()
    print("GPU tensor转numpy: ", arr_gpu)

#### 4.3 随机张量生成API汇总

In [ ]:
torch.manual_seed(42)
t_rand = torch.rand(2,3)
t_randn = torch.randn(2,3)
t_randint = torch.randint(0,10,(2,3))
t_perm = torch.randperm(5)
print("rand:\n", t_rand)
print("randn:\n", t_randn)
print("randint:\n", t_randint)
print("randperm:", t_perm)

#### 4.4 张量序列化：save & load

In [ ]:
save_path = "tensor_demo.pt"
t_save = torch.tensor([1.0,2.0,3.0])
torch.save(t_save, save_path)
print(f"张量已保存至 {save_path}")

t_load = torch.load(save_path, map_location=device)
print("加载出来的张量:", t_load)
print("设备:", t_load.device)

if os.path.exists(save_path):
    os.remove(save_path)
    print("临时文件已清理")

### 五、Tensor 完整属性与方法分类
#### 5.1 核心内置属性

**通用属性**（ndarray 也有）：

| 属性/方法 | 作用 |
|-----------|------|
| `shape` | 张量的形状，各维度的大小（返回 torch.Size） |
| `dim()` | 张量的维度数（几维） |
| `dtype` | 张量的数据类型（float32、int64 等） |
| `numel()` | 元素总数（各维度大小的乘积） |

**独有属性**（Tensor 才有，ndarray 没有）：

| 属性 | 作用 |
|------|------|
| `device` | 张量所在的设备（CPU / GPU），决定运算在哪执行 |
| `requires_grad` | 梯度追踪开关：True 才会被 autograd 记录、参与求导 |
| `is_leaf` | 是否为叶子张量（用户直接创建，而非运算产生）——决定梯度是否默认保留 |
| `grad` | 反向传播结束后存放该张量的梯度，与张量同 shape |
| `grad_fn` | 记录"这个张量是哪个算子算出来的"，是反向遍历的入口 |
| `data` | 老式接口：取底层数据、不追踪梯度（历史遗留，现推荐用 `detach()`） |

#### numel() 元素总数

In [ ]:
weight_t = torch.randn(2, 3, requires_grad=True, dtype=torch.float32)
print("权重张量:")
print(f"  shape: {weight_t.shape}, dim: {weight_t.dim()}, dtype: {weight_t.dtype}")
print(f"  device: {weight_t.device}")
print(f"  requires_grad: {weight_t.requires_grad}")
print(f"  is_leaf: {weight_t.is_leaf}")
print(f"  grad: {weight_t.grad}")
print(f"  grad_fn: {weight_t.grad_fn}")

In [ ]:
# numel() 演示
img_t = torch.randn(8, 3, 32, 32)
print(f"shape: {img_t.shape}")
print(f"dim(): {img_t.dim()}")
print(f"numel(): {img_t.numel()}")

total = 1
for s in img_t.shape:
    total *= s
print(f"手动验算: {total}")

#### 5.2 方法分类速查表

> 上一节列了属性（"是什么"），这一节列方法（"能做什么"）。Tensor 的方法按功能分为以下几类，各类的动手演示散见本章对应小节或前文，这里给出一张总览，方便查阅。

| 类别 | 代表方法 | 作用 | 详见 |
|------|---------|------|------|
| 创建 | `torch.tensor` / `zeros` / `ones` / `arange` / `clone` | 从数据、填充值、序列创建，或深拷贝已有张量 | 2.3 |
| 形状变换 | `reshape` / `view` / `transpose` / `permute` / `unsqueeze` / `squeeze` / `flatten` | 改变形状或维度排列 | 2.6 |
| 索引切片 | `t[i]` / `t[:, 0]` / 布尔索引 / 花式索引 | 取出张量的子集 | 2.7 |
| 数学运算 | `+ - * /` / `matmul` / `mm` / `pow` | 逐元素运算与矩阵运算 | 2.4 / 2.5 |
| 归约统计 | `sum` / `mean` / `max` / `min` / `std` / `argmax` / `topk` | 沿维度聚合、求极值、取前 K | 5.6 |
| 拼接分割 | `cat` / `stack` / `split` / `chunk` | 合并或拆分张量 | 5.7 |
| 类型转换 | `to(dtype)` / `float()` / `int()` / `long()` / `bool()` | 切换数据类型 | 5.8 |
| 设备迁移 | `to(device)` / `cpu()` / `cuda()` | 在 CPU / GPU 之间搬运张量 | 5.5 |
| 梯度相关 | `backward` / `detach` / `retain_grad` / `requires_grad_` / `register_hook` / `grad.zero_()` | 操控自动求导：反向、切断、保留、清梯度 | 第03章 |
| 序列化 | `torch.save` / `torch.load` | 张量存盘与加载 | 4.4 |

> 说明：梯度相关方法这里先认识"有哪些、干什么"，完整机制与组合规则是下一章《torch.autograd 核心详解》的主线。

In [ ]:
# 5.3 反向传播后查看 grad
loss = weight_t.sum()
loss.backward()
print("反向传播后 grad:\n", weight_t.grad)

In [ ]:
# 5.4 维度变换
t_ori = torch.tensor([[1,2,3],[4,5,6]])
print(f"原始 shape: {t_ori.shape}")
print(f"reshape(3,2): {t_ori.reshape(3,2).shape}")
print(f"transpose(0,1): {t_ori.transpose(0,1).shape}")
print(f"unsqueeze(0): {t_ori.unsqueeze(0).shape}")

In [ ]:
# 5.5 设备迁移
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t_dev = torch.tensor([1,2,3])
t_moved = t_dev.to(dev)
print(f".to() 迁移设备: {t_moved.device}")

#### 5.6 张量归约与统计

In [ ]:
t = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=torch.float32)
print(f"sum: {t.sum().item()}")
print(f"mean: {t.mean().item():.2f}")
print(f"max: {t.max().item()}")
print(f"min: {t.min().item()}")
print(f"std: {t.std().item():.2f}")

# 指定维度归约
t3 = torch.randn(2, 3, 4)
print(f"\nsum(dim=0): {t3.sum(dim=0).shape}")
print(f"sum(dim=1): {t3.sum(dim=1).shape}")
print(f"sum(dim=2): {t3.sum(dim=2).shape}")

# argmax
t2 = torch.tensor([[1, 5, 3], [8, 2, 9]])
print(f"\nargmax(dim=0): {t2.argmax(dim=0)}")
print(f"argmax(dim=1): {t2.argmax(dim=1)}")

# topk
values, indices = t.topk(k=2, dim=1)
print(f"\ntopk(2):\n{values}")

#### 5.7 张量拼接与分割

In [ ]:
# cat vs stack
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[5, 6], [7, 8]])

cat_0 = torch.cat([a, b], dim=0)
cat_1 = torch.cat([a, b], dim=1)
print(f"cat(dim=0): {cat_0.shape}")
print(f"cat(dim=1): {cat_1.shape}")

stack_0 = torch.stack([a, b], dim=0)
stack_1 = torch.stack([a, b], dim=1)
print(f"stack(dim=0): {stack_0.shape}")
print(f"stack(dim=1): {stack_1.shape}")

In [ ]:
# split vs chunk
t = torch.arange(12).reshape(6, 2)
split_list = t.split(split_size=2, dim=0)
print(f"split(2): {len(split_list)} 个张量")

chunk_list = t.chunk(chunks=3, dim=0)
print(f"chunk(3): {len(chunk_list)} 个张量")

large_batch = torch.randn(64, 3, 224, 224)
sub_batches = large_batch.split(16, dim=0)
print(f"\n大 batch: {large_batch.shape} → 拆成 {len(sub_batches)} 个 {sub_batches[0].shape}")

#### 5.8 张量类型转换

In [ ]:
t = torch.tensor([1, 2, 3])
print(f"原始: {t.dtype}")
print(f"to(float32): {t.to(torch.float32).dtype}")
print(f"to(float16): {t.to(torch.float16).dtype}")
print(f"to(bool): {t.to(torch.bool).dtype}")

# 快捷方法
t_f = torch.tensor([1, 2, 3], dtype=torch.float32)
print(f"\n.int(): {t_f.int().dtype}")
print(f".long(): {t_f.long().dtype}")
print(f".double(): {t_f.double().dtype}")

---

### 附录一：输入样本不求导，为什么还必须转 Tensor？

> 本附录旨在回答一个核心追问：**"不像网络参数，输入样本（图像/文本/时序）毋须计算梯度，为什么非得转 Tensor？ndarray 明明计算能力也够。"**

#### 核心答案

**输入样本虽然不求导（`requires_grad=False`），但 Tensor 拥有 ndarray 完全缺失的四大隐性能力。这些能力是 PyTorch C++ 引擎管理计算图节点所必需的——与是否求导无关。**

#### Tensor 的四大隐性能力（ndarray 完全缺失）

> **符号说明**：✅ = 是/具备，❌ = 否/不具备

| 隐性能力 | ndarray 有吗？ | 为什么输入样本需要它？ |
|---------|---------------|---------------------|
| **`grad_fn` 槽位** | ❌ | 作为计算图节点，必须能被 `next_edges` 引用 |
| **`_version` 版本控制** | ❌ | 前向时记录版本，确保数据未被意外修改 |
| **CUDA 流句柄** | ❌ | 能在 GPU 上执行计算 |
| **设备标识** | ❌ | 能与权重在同一设备上运算 |

---

#### 能力1：`grad_fn` 槽位 —— 计算图节点的"身份标识"

即使输入不求导，它**仍然是计算图中的一个节点**。`y = X @ W` 产生的 `y.grad_fn`，其 `next_edges` 中会包含指向 `X` 的引用。反向传播时，引擎需要遍历这个引用链——如果 `X` 是 ndarray，引擎无法解析。

**代码验证：**

In [ ]:
# 验证 grad_fn 槽位
print("=" * 60)
print("能力1：grad_fn 槽位 —— 计算图节点的身份标识")
print("=" * 60)

print("\n1. Tensor 作为输入（正确）：")
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=False)
w = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)
y = x * w
loss = y.sum()
print(f"   loss.grad_fn.next_functions: {loss.grad_fn.next_functions}")
print("   ⬆ next_functions 中包含了 x 的引用（虽然 x 不求导）")
print(f"   ✅ Tensor 有 grad_fn 槽位")

print("\n2. ndarray 作为输入（错误）：")
arr = np.array([1.0, 2.0, 3.0])
try:
    arr.grad_fn
except AttributeError as e:
    print(f"   ❌ ndarray 无 grad_fn: {e}")

print("\n3. 如果 x 是 ndarray，C++ 引擎无法解析引用：")
try:
    # 直接尝试用 ndarray 参与运算
    y_err = arr * w
    print("   ⚠️ 意外成功")
except TypeError as e:
    print(f"   ❌ 报错: {e}")
    print("   → 因为 C++ 算子只认 Tensor 类型")

print("\n💡 结论：即使输入不求导，它也必须是计算图节点，")
print("   而 grad_fn 槽位是节点的身份标识——ndarray 没有。")

---

#### 能力2：`_version` 版本控制 —— 数据一致性保护

前向传播时，每个 Tensor 的 `_version` 被记录。反向传播时，如果 Tensor 被就地修改（`_version` 变化），引擎会报错——**防止你用错误的数据算梯度**。

输入样本虽然不求导，但如果在前向和反向之间被修改，会导致权重梯度算错（因为 `dL/dW` 依赖 `X` 的数值）。

**代码验证：**

In [ ]:
# 验证 _version 版本控制
print("=" * 60)
print("能力2：_version 版本控制 —— 数据一致性保护")
print("=" * 60)

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=False)
w = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)
y = x * w
loss = y.sum()

print(f"1. 初始 x._version: {x._version}")

print("\n2. 执行 x.add_(10) 就地修改：")
x.add_(10)
print(f"   修改后 x: {x}")
print(f"   修改后 x._version: {x._version} (自增 +1)")

print("\n3. 尝试反向传播：")
try:
    loss.backward()
    print("   ⚠️ 意外成功")
except RuntimeError as e:
    print(f"   ❌ 预期的 RuntimeError:\n   {e}")

print("\n4. ndarray 是否有 _version？")
arr = np.array([1.0, 2.0, 3.0])
try:
    arr._version
except AttributeError as e:
    print(f"   ❌ ndarray 无 _version: {e}")

print("\n💡 结论：即使 x 不求导，修改它也会导致反向传播失败，")
print("   因为权重的梯度依赖于 x 的数值。")

---

#### 能力3：CUDA 流句柄 —— 在 GPU 上运行

ndarray 完全无法运行在 GPU 上。而深度学习训练几乎必须用 GPU 加速。

**代码验证：**

In [ ]:
# 验证 CUDA 流句柄
print("=" * 60)
print("能力3：CUDA 流句柄 —— GPU 运行能力")
print("=" * 60)

if torch.cuda.is_available():
    print("1. Tensor 可以放 GPU：")
    x_gpu = torch.tensor([1.0, 2.0, 3.0], device='cuda')
    print(f"   x_gpu.device: {x_gpu.device}")
    print(f"   ✅ Tensor 有 CUDA 流句柄")

    print("\n2. ndarray 无法放 GPU：")
    arr = np.array([1.0, 2.0, 3.0])
    try:
        arr.to('cuda')
    except AttributeError as e:
        print(f"   ❌ ndarray 无 .to() 方法: {e}")
else:
    print("当前环境无可用 GPU，跳过 CUDA 验证")
    print("\n但在实际深度学习训练中，GPU 是标配。")

print("\n💡 结论：ndarray 无法运行在 GPU 上，")
print("   而深度学习训练必须依赖 GPU 加速。")

---

#### 能力4：设备标识 —— 与权重保持设备一致性

当权重在 GPU 上时，输入必须在同一设备上。ndarray 没有设备标识，无法与 GPU Tensor 混合运算。

**代码验证：**

In [ ]:
# 验证设备标识
print("=" * 60)
print("能力4：设备标识 —— 设备一致性")
print("=" * 60)

if torch.cuda.is_available():
    print("1. 权重在 GPU 上，输入也用 Tensor 放 GPU（正确）：")
    w_gpu = torch.tensor([0.5, 0.5, 0.5], device='cuda', requires_grad=True)
    x_gpu = torch.tensor([1.0, 2.0, 3.0], device='cuda', requires_grad=False)
    y_gpu = x_gpu * w_gpu
    print(f"   ✅ 成功执行，y 在 {y_gpu.device}")

    print("\n2. 权重在 GPU 上，输入用 ndarray（错误）：")
    x_arr = np.array([1.0, 2.0, 3.0])
    try:
        y_err = torch.mul(x_arr, w_gpu)
        print("   ⚠️ 意外成功")
    except TypeError as e:
        print(f"   ❌ 报错: {e}")
        print("   → ndarray 没有设备标识，无法与 GPU Tensor 运算")

    print("\n3. 正确做法：先转 Tensor，再迁移到 GPU")
    x_t_cpu = torch.from_numpy(x_arr)
    x_t_gpu = x_t_cpu.to('cuda')
    y_correct = x_t_gpu * w_gpu
    print(f"   ✅ 成功，y 在 {y_correct.device}")
else:
    print("当前环境无可用 GPU，跳过 CUDA 验证")

print("\n💡 结论：ndarray 没有设备标识，无法与 GPU Tensor 协同运算。")

---

### 附录二：`clone()` 完整专题

> 本附录完整讲解 `clone()` 的原理、透传机制、6 大应用场景及方法选择指南。所有代码均为独立可运行的 cell。

#### 一、核心概念：什么是"梯度流"？

梯度流是**反向传播时误差信号从输出层一路传回输入层的那条路径**：

```
loss  →  z  →  y  →  x * w1  →  x
               →            →  w1
               →  w2
```

`loss.backward()` 执行时，PyTorch 沿着 `grad_fn` 链反向走。如果链断了，梯度就到不了某些参数。

---

#### 二、`clone()` 的底层机制：透传节点

`clone()` 在计算图中**插入了一个 `CloneBackward` 节点**：

```
原始链:  x → MulBackward → y
                    ↑
                    w1

clone后: x → MulBackward → y → CloneBackward → y_clone
                    ↑
                    w1
```

- **前向**：复制数据
- **反向**：`CloneBackward` 把梯度**原样传回**（透传）

#### 透传验证：两层网络场景

In [ ]:
# 透传验证：标量两层网络
x = torch.tensor(2.0)                    # 数据，不求导
w1 = torch.tensor(3.0, requires_grad=True)
w2 = torch.tensor(4.0, requires_grad=True)

y = x * w1                               # y = 6.0
y_clone = y.clone()                      # 插入 CloneBackward
z = y_clone * w2                         # z = 24.0
loss = z.sum()
loss.backward()

print(f"w1.grad = {w1.grad}")            # dz/dw1 = x * w2 = 2.0 * 4.0 = 8.0
print(f"w2.grad = {w2.grad}")            # dz/dw2 = y = 6.0
print("\n✅ 所有梯度都正确到达！CloneBackward 是透传节点。")

---

#### 三、三种"复制"方式对比
> [<a href="http://127.0.0.1:5500/html/02_tensor_06_copy.html" target="_blank">👉 在浏览器打开 三种复制方式图解</a>](http://127.0.0.1:5500/html/02_tensor_06_copy.html)


> **符号说明**：✅ = 是/具备，❌ = 否/不具备

| 操作 | 数据是否独立 | grad_fn | 梯度能否流回原对象 |
|------|------------|---------|------------------|
| `y_clone = y.clone()` | ✅ | `CloneBackward` | ✅ |
| `y_detach = y.detach()` | ❌（共享） | `None` | ❌ |
| `y_new = torch.tensor(y)` | ✅ | `None`（叶子） | ❌ |

In [ ]:
# 三种复制方式代码验证
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2

print("=" * 50)
print("1. clone()：数据独立 + 保留梯度")
y_clone = y.clone()
print(f"   数据独立: {y_clone.data_ptr() != y.data_ptr()}")
print(f"   grad_fn: {y_clone.grad_fn is not None}")

print("\n2. detach()：数据共享 + 切断梯度")
y_detach = y.detach()
print(f"   数据独立: {y_detach.data_ptr() != y.data_ptr()}")
print(f"   grad_fn: {y_detach.grad_fn is not None}")

print("\n3. torch.tensor()：数据独立 + 新建叶子")
y_new = torch.tensor(y.tolist())
print(f"   数据独立: {y_new.data_ptr() != y.data_ptr()}")
print(f"   grad_fn: {y_new.grad_fn is not None}")

print("\n✅ 只有 clone() 做到'数据独立 + 保留梯度历史'")

---

#### 四、6 大必须用 `clone()` 的场景

| 场景 | 核心问题 | clone() 的贡献 | 不用会怎样 |
|------|---------|---------------|----------|
| **多任务学习** | 需要分别拿到各支路梯度 | 每条支路独立 grad 槽位 | 梯度累加到同一张量，难以分别观测各支路 |
| **梯度检查点相关** | 需保留可回溯的中间张量 | clone 保留数据与梯度路径 | 只剩引用，原值可能被 in-place 改写；注意官方 checkpoint 靠重算而非 clone |
| **安全修改** | 改副本不改原图 | 原 x 的梯度不变 | RuntimeError |
| **数据增强** | 多版本独立 | 每个版本独立，梯度都能回传 | 原始数据被污染 |
| **知识蒸馏** | 固定教师输出 | 独立副本，切断教师梯度 | 教师参数被误更新 |
| **梯度监控** | 拦截中间梯度 | 在独立副本上注册 hook | hook 可能被覆盖 |

In [ ]:
# 场景1：多任务学习 —— clone() 让两条支路梯度互不覆盖
print("=" * 50)
print("场景1：多任务学习")
print("=" * 50)

class SharedBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 16, 3, padding=1)
    def forward(self, x):
        return self.conv(x)

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(16*32*32, 10)
    def forward(self, x):
        return self.fc(x.flatten(1))

class Segmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(16, 1, 3, padding=1)
    def forward(self, x):
        return self.conv(x)

backbone = SharedBackbone()
cls_head = Classifier()
seg_head = Segmenter()

feature = backbone(torch.randn(2, 3, 32, 32))

# ✅ 用 clone() 创建两个独立分支
cls_feat = feature.clone()
seg_feat = feature.clone()

cls_out = cls_head(cls_feat)
seg_out = seg_head(seg_feat)

loss = cls_out.sum() + seg_out.sum()
loss.backward()

print(f"cls_feat.grad 独立存在: {cls_feat.grad is not None}")
print(f"seg_feat.grad 独立存在: {seg_feat.grad is not None}")
print("✅ 两条支路各算各的梯度，互不覆盖")
print("\n⚠️ 若不用 clone，两条支路共享同一张量、梯度会累加到一起，无法分别取得各支路的梯度")

In [ ]:
# 场景2：梯度检查点 —— 官方机制是"时间换显存"，并非靠 clone() 实现（概念澄清）
print("=" * 50)
print("场景2：梯度检查点（概念澄清）")
print("=" * 50)

from torch.utils.checkpoint import checkpoint

def segment(x):
    # 模拟一段中间激活很多的计算
    return (x ** 2).sum()

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
# checkpoint 前向不保存中间激活，反向时重算，梯度依然正确回传
z = checkpoint(segment, x, use_reentrant=False)
z.backward()
print(f"✅ 官方 checkpoint 反向重算，梯度正确回传到 x: {x.grad}")
print("💡 梯度检查点是'时间换显存'：中间激活不存、反向时重算；由 torch.utils.checkpoint 实现，")
print("   并不是靠 clone()。clone() 在此只保留一个可回溯的张量副本，不等同于 checkpoint。")

# 对比：detach() 会彻底切断梯度
x2 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y2 = x2.detach() ** 2
loss2 = y2.sum()
try:
    loss2.backward()
    print(f"   detach 后 x2.grad = {x2.grad}（为 None，梯度被切断）")
except RuntimeError as e:
    print(f"❌ detach() 导致梯度断链: {type(e).__name__}")
print("\n💡 三者区别：detach() 切断梯度；clone() 保留梯度路径；checkpoint 是显存优化机制。")

In [ ]:
# 场景3：安全 In-place 修改 —— clone() 避免 RuntimeError
print("=" * 50)
print("场景3：安全 In-place 修改")
print("=" * 50)

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2
loss = y.sum()

# ✅ 安全：克隆后修改
x_safe = x.clone()
x_safe.add_(10)
loss.backward()
print(f"✅ clone() 后安全修改，梯度正常传到 x: {x.grad}")
print("\n💡 直接修改原张量会报错：one of the variables needed for gradient...")

In [ ]:
# 场景4：数据增强 —— clone() 保留原始样本
print("=" * 50)
print("场景4：数据增强")
print("=" * 50)

def random_crop(img, size=28):
    h, w = img.shape[-2], img.shape[-1]
    top = torch.randint(0, h - size, (1,)).item()
    left = torch.randint(0, w - size, (1,)).item()
    return img[..., top:top+size, left:left+size]

def random_flip(img):
    if torch.rand(1) > 0.5:
        return img.flip(-1)
    return img

original = torch.randn(3, 32, 32)

view1 = original.clone()
view1 = random_crop(view1, 28)

view2 = original.clone()
view2 = random_flip(view2)

print(f"✅ 两个增强版本独立，原始图像未变: {original.shape}")
print(f"   view1 shape: {view1.shape}")
print(f"   view2 shape: {view2.shape}")
print("\n💡 若不用 clone，第一个增强会修改 original，第二个增强拿不到原始数据")

In [ ]:
# 场景5：知识蒸馏 —— clone().detach() 固定教师输出
print("=" * 50)
print("场景5：知识蒸馏")
print("=" * 50)

class TeacherModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 5)
    def forward(self, x):
        return self.fc(x)

class StudentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 5)
    def forward(self, x):
        return self.fc(x)

teacher = TeacherModel()
student = StudentModel()

x = torch.randn(4, 10)

teacher_logits = teacher(x)
teacher_target = teacher_logits.clone().detach()

student_logits = student(x)
loss = (student_logits - teacher_target).pow(2).sum()
loss.backward()

teacher_grad = any(p.grad is not None for p in teacher.parameters())
student_grad = any(p.grad is not None for p in student.parameters())
print(f"✅ 教师模型被冻结: {teacher_grad}")
print(f"✅ 学生模型有梯度: {student_grad}")
print("\n💡 若不用 clone().detach()，教师模型的梯度会被误计算和更新")

In [ ]:
# 场景6：梯度监控 —— clone() 在独立副本上注册 hook
print("=" * 50)
print("场景6：梯度监控")
print("=" * 50)

x = torch.randn(2, 10, requires_grad=True)
w1 = torch.randn(10, 20, requires_grad=True)
w2 = torch.randn(20, 5, requires_grad=True)

h1 = x @ w1
h1_monitor = h1.clone()

def gradient_hook(grad):
    print(f"   捕获到梯度，shape: {grad.shape}, mean: {grad.mean().item():.4f}")
    return grad

h1_monitor.register_hook(gradient_hook)

h2 = h1_monitor @ w2
loss = h2.sum()
loss.backward()

print("✅ 成功在中间层捕获梯度，且不影响原始计算图")
print("\n💡 用途：梯度爆炸检测、Grad-CAM 可视化、梯度裁剪等")

---

#### 五、方法选择速查表

> **符号说明**：✅ = 是/具备，❌ = 否/不具备

| 需求 | 推荐方法 | 数据独立 | 保留梯度 | 梯度能流回原对象 |
|------|---------|---------|---------|----------------|
| 需要独立数据 + 保留梯度历史 | `clone()` | ✅ | ✅ | ✅ |
| 需要独立数据 + 切断梯度 | `clone().detach()` | ✅ | ❌ | ❌ |
| 需要共享数据 + 切断梯度 | `detach()` | ❌ | ❌ | ❌ |
| 需要零拷贝视图 | `view()` / `transpose()` | ❌ | ✅* | ✅* |
| 从 Python 数据创建 | `torch.tensor()` | ✅ | ❌ | ❌ |
| 从 NumPy 零拷贝创建 | `torch.from_numpy()` | ❌ | ❌ | ❌ |

> \* `view()` 和 `transpose()` 共享内存，保留梯度历史（因为是同一份数据）。

---

### 附录三：`.data` 属性专题（历史遗留）

> 本附录完整讲解 `.data` 属性的来龙去脉、核心用途、安全性问题及现代替代方案。**结论先行：新代码永远不要用 `.data`，用 `.detach()` 或 `torch.no_grad()` 替代。**

#### 一、什么是 `.data` ？

`.data` 是 PyTorch 早期版本的遗留属性，用于**获取一个与原始张量共享内存、但不追踪梯度的纯数据视图**。

#### 历史背景

在 PyTorch 0.4.0 之前，`Variable`（负责记录计算历史）和 `Tensor`（负责存储数据）是两个独立的对象。`.data` 是 `Variable` 的一个属性，用来**取出它包裹的底层 `Tensor`**。

PyTorch 0.4.0 合并了 `Variable` 和 `Tensor`，但 `.data` 作为兼容属性保留了下来——它的原始使命已经不存在了。

#### 二、`.data` 的核心用途

`.data` 的设计目的是：**在不触发 Autograd 追踪的前提下，直接读写张量的底层数值数据**。

#### 典型场景：参数原地更新

参数更新是**数值修改**，不是张量**运算**，不应被记录为计算图节点。如果写成 `w = w - lr * w.grad`，会创建新张量，丢失叶子节点身份，计算图断裂。

#### 为什么现在用不着了？

1. **优化器自动完成**：`torch.optim.SGD` 等优化器内部已封装好参数更新逻辑，用户无需手动操作。
2. **`torch.no_grad()` 更安全**：语义清晰，且能感知版本变化。

> `.data` 现在是**底层开发工具**，普通训练任务中完全用不到。

#### 三、`.data` 的三大特点

| 特点 | 说明 |
|:---|:---|
| **共享内存** | `.data` 返回与原始张量共享内存的视图，修改会同步影响原张量 |
| **不追踪梯度** | `.data` 视图的 `requires_grad` 始终为 `False` |
| **绕过 Autograd** | 直接触碰底层数据内存，完全不经过 Autograd 系统 |


In [ ]:
# .data 三大特点演示：共享内存 + 不追踪梯度
print(".data 三大特点演示")
print("=" * 40)

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"1. 原始张量 x: {x}, requires_grad={x.requires_grad}")

x_data = x.data
print(f"2. x.data 视图: {x_data}, requires_grad={x_data.requires_grad}")

x_data[0] = 999
print(f"3. 修改 x.data[0]=999 后，原 x 同步变化: {x}")
print("   ✅ 共享内存 + 不追踪梯度")

#### 老式 vs 现代写法对比

```python
# 老式写法（不推荐）
w.data -= 0.1 * w.grad

# 现代写法（推荐）
with torch.no_grad():
    w -= 0.1 * w.grad
```

#### 四、为什么不推荐用 `.data`？

#### 核心问题：不安全——修改不会被 Autograd 感知

`.data` 直接从底层存储中取出数据指针，**完全绕开了 Autograd 的版本控制机制**。修改 `.data` 视图时，Autograd 完全不知情，梯度计算使用错误数据，但不会抛出任何异常。

> 这是最危险的地方：**不报错，错误静默传播**。调试时极难排查。

**代码验证：.data 导致静默错误**

In [ ]:
# .data 静默错误演示
print(".data 静默错误演示")
print("=" * 40)

a = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"a: {a}")

out = a.sigmoid()
print(f"out: {out}")

c = out.data                       # 与 out 共享内存
c.zero_()                          # 修改了 out 的数据
print(f"c.zero_() 后 out: {out}")

out.sum().backward()
print(f"a.grad = {a.grad}")         # 全是 0，错误但不报错！
print("⚠️ 梯度全是 0，但没有任何报错！错误静默传播！")

#### 五、`.data` vs `.detach()` 对比

| 维度 | `.data` | `.detach()` |
|:---|:---|:---|
| 共享内存 | ✅ 是 | ✅ 是 |
| 不追踪梯度 | ✅ 是 | ✅ 是 |
| 修改是否被 Autograd 感知 | ❌ 否（静默错误） | ✅ 是（主动报错） |
| 官方推荐 | ❌ 不推荐 | ✅ 推荐 |

#### `detach()` 如何主动报错？

`detach()` 返回的视图会**保留版本计数器的引用**。原地修改会触发版本号变更，反向传播时检测到版本不匹配，立即抛出 `RuntimeError`。

**代码验证：detach() 主动报错**

In [ ]:
# detach() 主动报错演示
print("detach() 主动报错演示")
print("=" * 40)

a = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = a.sigmoid()
print(f"out._version = {out._version} (初始版本)")

c = out.detach()
c.zero_()
print(f"c.zero_() 后，out._version = {out._version} (自增 1)")

try:
    out.sum().backward()
    print("⚠️ 意外成功")
except RuntimeError as e:
    print(f"✅ 预期的报错:\n   {e}")

print("\n💡 这个 RuntimeError 是 detach() 故意报的，目的是暴露问题。")

#### 六、现代替代方案

| 场景 | 旧写法（不推荐） | 新写法（推荐） |
|:---|:---|:---|
| 参数更新 | `p.data -= lr * p.grad` | `with torch.no_grad(): p -= lr * p.grad` |
| 取出纯数据视图 | `x.data` | `x.detach()` |
| 修改数据 | `x.data[0] = 999` | `with torch.no_grad(): x[0] = 999` |
| 转 NumPy | `x.data.numpy()` | `x.detach().numpy()` |

#### 七、重要结论

| 结论 | 说明 |
|:---|:---|
| **新代码不要用 `.data`** | 已被 `.detach()` 和 `torch.no_grad()` 完全替代 |
| **看到老代码中的 `.data`** | 知道它是历史遗留，识别其风险 |
| **面试考点** | `.data` 和 `.detach()` 的区别是高频问题 |
| **理解 PyTorch 演进** | 知道 `Variable` → `Tensor` 的历史演变 |

> **一句话总结**：`.data` 是 PyTorch 早年设计的"漏洞"——它可以绕过 Autograd 修改数据且不被感知，导致静默错误。现在已被 `.detach()` 完全替代。